In [2]:
# --- Step 1: Load bizitobs_l2c/H/short data using datasets package ---
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from toto.data.util.dataset import MaskedTimeseries
from toto.inference.forecaster import TotoForecaster
from toto.model.toto import Toto
import torch
from datetime import datetime, timedelta

import matplotlib
import matplotlib.pyplot as plt
from collections import defaultdict
import pickle
import numbers
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
print(pd.__version__)

2.2.3


In [5]:
# # experiment_name = 'lag_amount_24'
# # ============================ SimpleTSbench datasets ============================
# data_mode = 'simpletime'  # 'gift-eval' or 'simpletime'
# data_types = [
#     "univariate", 
#     "multivariate", 
#     'multivariate_lagged'
#     ]

# all_data =  [
#     "constant",
#     "exponential",
#     "linear",
#     "noise_patch",
#     "normal",
#     "poisson",
#     "random_walk",
#     "sawtooth",
#     "sine",
#     "staircase"
# ]
# dataset_dir = "../../data/simpletime/datasets"

# ============================ GIFT-eval datasets ============================
data_mode = 'gift-eval'  # 'gift-eval' or 'simpletime'
data_types = [
    'gift_eval'
]

all_data =  [
    # "bitbrains_fast_storage/5T",
    # "bitbrains_fast_storage/H",
    # "bitbrains_rnd/5T",
    # "bitbrains_rnd/H",
    # "bizitobs_application/10S", # remove
    # "bizitobs_l2c/5T",
    "bizitobs_l2c/H",
    # "bizitobs_service/10S", # remove
    # "ett1/15T",
    # "ett1/D",
    # "ett1/H",
    # "ett1/W",
    # "ett2/15T",
    # "ett2/D",
    # "ett2/H",
    # "ett2/W",
    # "jena_weather/10T",
    # "jena_weather/D",
    # "jena_weather/H",
    # "electricity/15T",
    # "electricity/D",
    # "electricity/H",
    # "electricity/W",
    # "hierarchical_sales/D",
    # "hierarchical_sales/W",
    # "kdd_cup_2018_with_missing/D",
    # "kdd_cup_2018_with_missing/H",
    # "LOOP_SEATTLE/5T",
    # "LOOP_SEATTLE/D",
    # "LOOP_SEATTLE/H",
    # "M_DENSE/D",
    # "M_DENSE/H",
    # "SZ_TAXI/15T",
    # "SZ_TAXI/H",
    # "saugeenday/D",
    # "saugeenday/M",
    # "saugeenday/W",
    # "solar/10T",
    # "solar/D",
    # "solar/H",
    # "solar/W",
    # "us_births/D",
    # "us_births/M",
    # "us_births/W",

]

freq_map = {
            "10S": 10,           # 10 seconds → 10 seconds
            "5T": 5 * 60,        # 5 minutes → 300 seconds
            "10T": 10 * 60,      # 10 minutes → 600 seconds
            "15T": 15 * 60,      # 15 minutes → 900 seconds
            "H": 1 * 3600,       # 1 hour → 3,600 seconds
            "D": 1 * 86400,      # 1 day → 86,400 seconds
            "W": 7 * 86400,      # 1 week → 604,800 seconds
            "M": 30 * 86400,      # 1 month → 2,592,000 seconds
        }

REPO_ROOT = os.getenv('REPO_ROOT')

dataset_dir = f"{REPO_ROOT}/data" # for gift-eval
# dataset_dir = "/home/roc/01_projects/11_time-series/timeseries-research/data"

samples_per_batch = 16
forecast_log = {}
metrics_log = []
context_pct = 0.8
prediction_pct = 0.2
context_length_cap = 512
prediction_length_cap = 96 # [96, 192, 336, 720]
UNIVAR_N = 10  # For some univariate datasets where the entire series is broken into multiple rows, only use the first N rows
DEVICE = "cuda"

def load_data(dataset_name, data_type, freq='H', data_mode='simpletime'):
    global dataset_dir
    # Path to Arrow file directory (datasets expects a directory, not a file)
    if data_mode == 'simpletime':
        DATASET_DIR = f"{dataset_dir}/{data_type}/{dataset_name}"
    else:
        DATASET_DIR = f"{dataset_dir}/{data_type}/{dataset_name}/{freq}"
    # Load the dataset using the datasets package
    ds = load_dataset('arrow', data_files={"train": os.path.join(DATASET_DIR, "data-00000-of-00001.arrow")})
    # Convert to pandas DataFrame for inspection
    df = ds['train'].to_pandas()
    return df




In [7]:
df = load_data('bizitobs_l2c', 'gift_eval', 'H', 'gift_eval')

In [10]:
target = df['target'][0]

In [11]:
target.shape

(7,)

In [12]:
target[0].shape

(2664,)